# Lab 2 · What an average hides
## Data Science Fundamentals — Week 2: Describing Data
**Mohammad Sabokrou · New Uzbekistan University**

**Your job:** use evidence to explain data and make a defensible decision. The Python is provided; programming complexity earns no extra credit.

**Name:** …   **Student ID:** enter in the setup cell below.

### How to work
1. Open this notebook in Google Colab or Jupyter. Run the setup with your own ID.
2. Work from top to bottom. Where asked, write a prediction **before** running the next cell. Keep the prediction even if it is wrong; explain what you learned.
3. Double-click an **Answer** cell to write. Use your own generated numbers and refer to specific plots. A number without an interpretation is incomplete.
4. Run the supplied code; change only the explicitly labelled settings. You are responsible for understanding and explaining the results you submit.
5. Keep the same ID throughout. Restart and run all cells after finishing; save the notebook with outputs.

**Suggested pace:** setup 5 min; sections 1–8: 10, 15, 15, 15, 15, 10, 15, 15 min; final check 5 min (120 min total). Finish the written reflection after class if needed.

### What varies between students?
Your ID determines the data, eight-number exercise, selected record, histogram settings, and decision scenario. The concepts and workload stay comparable. Reusing an ID gives the same case. Different IDs are not a guarantee that every individual number differs.

All records are **synthetic teaching data**, not real university records. There are no downloads, joins, or missing code blocks to debug.

**Evidence standard:** answers should contain a claim, a number or plot observation from your case, and a reason or limitation. We assess statistical reasoning, appropriate methods, and reproducible evidence—not Python fluency.

In [1]:
# SETUP — edit only STUDENT_ID. Integers and quoted IDs both work.
STUDENT_ID = "240116"
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

STUDENT_ID = str(STUDENT_ID).strip()
if not (STUDENT_ID.isascii() and STUDENT_ID.isdigit() and 6 <= len(STUDENT_ID) <= 12):
    raise ValueError("Replace YOUR_ID with your own 6–12 digit student ID, then run again.")
seed = int.from_bytes(hashlib.sha256(('FDS-Lab2-v1:' + STUDENT_ID).encode()).digest()[:8], 'big')
rng = np.random.default_rng(seed)
case_code = hashlib.sha256(('FDS-Lab2-v1:' + STUDENT_ID).encode()).hexdigest()[:10]
plt.rcParams.update({'figure.figsize': (8, 4), 'axes.spines.top': False, 'axes.spines.right': False})
pd.options.display.float_format = '{:.3f}'.format
N = 240
programme = rng.choice(['A', 'B'], N, p=[.55, .45])
entry = np.clip(rng.normal(rng.uniform(62,68), rng.uniform(8,11), N), 0, 100)
minutes = rng.lognormal(rng.uniform(4.1,4.4), .85, N)
# Two different teaching groups: their score patterns intentionally differ.
assessment = np.clip(rng.normal(np.where(programme == 'A', 43, 78), 4.5), 0, 100)
satisfaction = rng.choice([1.,2.,3.,4.,5.], N, p=[.08,.12,.2,.35,.25])
satisfaction[rng.choice(N, size=int(rng.integers(35,61)), replace=False)] = np.nan
students = pd.DataFrame({'record_id':np.arange(1,N+1), 'programme':programme,
    'entry_score':entry, 'weekly_minutes':minutes,
    'assessment_score':assessment, 'satisfaction':satisfaction})
small = np.sort(rng.choice(np.arange(35,86), 8, replace=False)).astype(float)
focus_index = int(np.argmax(entry))
coarse_bins = int(rng.choice([2,3,4]))
fine_bins = int(rng.choice([35,40,45]))
scenario = str(rng.choice(['typical use', 'total workload']))
print('Case:',case_code, '| ID:',STUDENT_ID, '| Final decision:',scenario)
print('Keep this case code in your submission. Setup complete.')

Case: b4b1cd8edf | ID: 240116 | Final decision: typical use
Keep this case code in your submission. Setup complete.


## 1 · Who and what are we describing? (10 min)
One row is one synthetic student observed in the same week. Scores are points out of 100; minutes are minutes per student in that week. Satisfaction is an **ordered category** from 1 (lowest) to 5 (highest); equal numerical steps need not represent equal changes in feeling. Programme A/B are category labels; record IDs are identifiers.

The next cell displays the data and counts. `count()` excludes missing values; `len()` counts rows. No missing value is replaced by zero.

In [2]:
display(students.head())
valid = students['satisfaction'].count()
print(f'Total rows: {len(students)}; valid satisfaction: {valid}; missing: {len(students)-valid}')
print(f'Response rate: {100*valid/len(students):.1f}%')
display(students['satisfaction'].value_counts().sort_index().rename('responses'))
print('Most common programme(s):', students['programme'].mode().tolist())

,record_id,programme,entry_score,weekly_minutes,assessment_score,satisfaction
0,1,B,61.833,124.324,71.875,5.000
1,2,A,66.870,79.073,46.337,NaN
2,3,A,55.379,18.657,46.961,3.000
3,4,A,51.270,82.677,38.870,1.000
4,5,A,56.982,110.907,42.106,5.000


Total rows: 240; valid satisfaction: 180; missing: 60
Response rate: 75.0%


,responses
satisfaction,
1.000,8
2.000,27
3.000,37
4.000,63
5.000,45


Most common programme(s): ['A']


### Your reasoning
**1A.** State the observational unit and the denominator for the satisfaction response rate. Why is the denominator for a mean of recorded satisfaction responses different?

**1B.** A colleague replaces missing satisfaction with zero. Explain why this changes the meaning. Can the responses establish what *all* students feel?

**1C.** Which is meaningful: the mode of programme, the mean of record ID, or the mean of programme coded A=1/B=2? Explain. For satisfaction, explain why the category frequency table adds information that a mean alone hides.

**Answer:**

**1A.** One row represents one student. The denominator for the satisfaction response rate is 240 because there are 240 students in total. There are 180 valid satisfaction responses, so the response rate is 75%. For the mean satisfaction, we use only the 180 students who actually gave a response. The 60 missing values are not counted.

**1B.** Replacing missing satisfaction with zero is not correct because zero does not mean “no answer” in this data. It would make the results look like some students had very low satisfaction when they actually gave no response. Also, we cannot say what all students feel because 60 out of 240 students did not answer the satisfaction question.

**1C.** The mode of programme is meaningful because it tells us which programme has more students. In this case, Programme A is the most common. The mean of record ID is not useful because the ID is only used to identify students. Coding A=1 and B=2 and finding the mean is also not meaningful because the numbers are just labels. The satisfaction frequency table is useful because it shows how many students chose each rating: 8 chose 1, 27 chose 2, 37 chose 3, 63 chose 4, and 45 chose 5. A mean alone would hide these details.


## 2 · Eight numbers, by hand first (15 min)
Run the next cell to obtain your eight numbers. Before running the checking cell, calculate their mean, median, **sample variance and sample SD** using a calculator if useful. Show intermediate steps; no Python writing is required.

Mean = sum / n. Deviations = value − mean. Sample variance = sum of squared deviations / (n−1). Sample SD = square root of sample variance.

In [3]:
print('Your eight scores:',small.tolist())

Your eight scores: [35.0, 56.0, 60.0, 61.0, 67.0, 74.0, 79.0, 85.0]


### Your reasoning
**2A.** Record: sum, n, mean, middle two scores, median, eight deviations, sum of squared deviations, sample variance, and sample SD. Include units.

**2B. Prediction:** if these eight scores were the entire population of interest, would the population SD be larger or smaller? Why?

**Answer:** Replace this text with your explanation.

In [ ]:
# CHECK ONLY AFTER YOUR HAND CALCULATIONS
s = pd.Series(small)
display(pd.DataFrame({'score':small,'deviation':small-small.mean(),
                     'squared deviation':(small-small.mean())**2}))
display(pd.Series({'mean':s.mean(),'median':s.median(),
    'sample variance':s.var(ddof=1),'pandas sample SD':s.std(),
    'NumPy population SD':np.std(small), 'NumPy sample SD':np.std(small,ddof=1)}))

### Your reasoning
**2C.** Compare your work with the output. Correct any error and explain its source. Why do the two default SD results differ? State which divisor answers each question; explain why variance has points² but SD has points.

**Answer:** Replace this text with your explanation.

## 3 · Middle half, extremes, and sensitivity (15 min)
For the eight scores, use **median of halves**: Q1 is the median of the lower four; Q3 the median of the upper four. For the larger dataset later, use pandas **linear interpolation**. Different stated conventions can legitimately give different quartiles.

Before running, compute the five-number summary and IQR of your eight scores. Then predict what happens if the largest score is accidentally multiplied by 10.

### Your reasoning
**3A.** Give min, Q1, median, Q3, max, and IQR by hand. Interpret the middle-half interval in a sentence.

**3B. Prediction:** which will change most after the error: mean or median? Will range, SD, and IQR change? Explain using which observations each measure depends on.

**Answer:** Replace this text with your explanation.

In [ ]:
def half_summary(a):
    a = np.sort(a)
    q1, q3 = np.median(a[:4]), np.median(a[4:])
    return pd.Series({'min':a.min(),'Q1':q1,'median':np.median(a),'Q3':q3,
        'max':a.max(),'mean':a.mean(),'range':np.ptp(a),'IQR':q3-q1,
        'sample variance':np.var(a,ddof=1),'sample SD':np.std(a,ddof=1)})
changed = small.copy(); changed[-1] *= 10
print('Changed scores:',changed.tolist())
display(pd.DataFrame({'original':half_summary(small),'with error':half_summary(changed)}))
print('Original pandas linear quartiles:',pd.Series(small).quantile([.25,.75]).to_dict())

### Your reasoning
**3C.** Compare predictions with results, citing at least two numerical changes. Why can the median and IQR stay stable here? Does that make them universally best?

**3D.** Explain any difference between hand and software quartiles. Why is the altered score impossible in this context, whereas a very large number of platform minutes may be valid?

**Answer:** Replace this text with your explanation.

## 4 · Look before summarising (15 min)
The next cell plots four numeric columns: entry score, weekly minutes, assessment score, and satisfaction. For the discrete ordinal satisfaction scale, each histogram bin represents one response category. Missing responses are excluded and the valid n is shown.

Read **centre, spread, tail direction, peaks/gaps, and unusual values**. `skew()` gives a sample skewness statistic; its sign is useful evidence, not a complete description of shape.

In [ ]:
columns = ['entry_score','weekly_minutes','assessment_score','satisfaction']
units = ['points','minutes per week','points','response category']
fig, axes = plt.subplots(2,2,figsize=(12,8))
for ax,col,unit in zip(axes.flat,columns,units):
    x=students[col].dropna()
    ax.hist(x,bins=np.arange(.5,6,1) if col=='satisfaction' else 18,
            color='#347c99',edgecolor='white')
    ax.set(title=f'{col} (valid n={len(x)})',xlabel=unit,ylabel='Student count')
plt.tight_layout(); plt.show()
display(students[columns[:3]].agg(['count','mean','median','std','skew']).T)
fig, axes = plt.subplots(1,3,figsize=(12,3.5))
for ax,b in zip(axes,[coarse_bins,12,fine_bins]):
    ax.hist(students.assessment_score,bins=b,color='#347c99',edgecolor='white')
    ax.set(title=f'{b} bins',xlabel='Assessment score (points)',ylabel='Student count',xlim=(0,100))
plt.tight_layout(); plt.show()

### Your reasoning
**4A.** Write one shape sentence for **each of the four columns**. Include plot evidence; do not merely list a skewness number.

**4B.** Find a column where mean and median disagree. Give both values and explain the mechanism. Which centre and spread would you report for a typical observation?

**4C.** Compare the three histogram bin settings. What is concealed or exaggerated? Choose one and justify it. Does changing bins change the underlying mean?

**4D.** Does a nearly zero skewness or similar mean/median establish a normal distribution? Use the assessment-score plot to explain.

**Answer:** Replace this text with your explanation.

## 5 · Boxplots, flags, and hidden groups (15 min)
Box = Q1 to Q3; line = median. Fences = Q1 − 1.5×IQR and Q3 + 1.5×IQR. Whiskers end at the most extreme **observed** values inside the fences; they are not the fences themselves. Dots are candidates for investigation, not automatic deletion.

Here pandas and the plotting convention use linearly interpolated quartiles. Group summaries use sample SD (`ddof=1`).

In [ ]:
x = students.weekly_minutes
q1,q3=x.quantile([.25,.75]); iqr=q3-q1
lower,upper=q1-1.5*iqr,q3+1.5*iqr
inside=x[(x>=lower)&(x<=upper)]
flagged=students[(x<lower)|(x>upper)]
display(pd.Series({'Q1':q1,'median':x.median(),'Q3':q3,'IQR':iqr,
    'lower fence':lower,'upper fence':upper,'lower whisker':inside.min(),
    'upper whisker':inside.max(),'flagged count':len(flagged)}))
display(flagged[['record_id','weekly_minutes']].head(6))
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].boxplot(x); axes[0].set(ylabel='Minutes per week',title='Weekly minutes',xticks=[])
axes[1].boxplot([students.loc[students.programme==g,'assessment_score'] for g in ['A','B']])
axes[1].set(xticks=[1,2],xticklabels=['A','B'],ylabel='Score (points)',title='Assessment by programme')
plt.tight_layout(); plt.show()
print('Overall assessment summary:')
display(students.assessment_score.agg(['count','mean','median','std']))
print('Programme summaries:')
display(students.groupby('programme').assessment_score.agg(['count','mean','median','std']))

### Your reasoning
**5A.** Using your Q1/Q3, show one fence calculation. State a whisker endpoint and explain why it differs from the corresponding fence. Describe a flagged record using its ID and minutes.

**5B.** Propose two checks before changing that record and one reason to retain it. What would you document if it were corrected? Do not delete it simply because it crosses a fence.

**5C.** Compare the overall assessment mean with both programme means and SDs. Would a single support session at the overall average meet both groups' needs? Give evidence and a better plan. Can these data prove that programme membership caused the difference?

**Answer:** Replace this text with your explanation.

### Same mean and SD, different students
The next two small classes have the same mean and sample SD but different shapes. Values are shifted for your case. Inspect them before answering.

In [ ]:
centre = int(round(small.mean()))
class_a = centre + np.array([-20,0,0,0,0,0,0,20])
class_b = centre + np.array([-10,-10,-10,-10,10,10,10,10])
for name,a in [('Class A',class_a),('Class B',class_b)]:
    print(name, a.tolist(), '| mean:',a.mean(),'| sample SD:',round(a.std(ddof=1),3))

### Your reasoning
**5D.** Sketch or describe each shape. Explain why the same mean and SD can lead to different tutoring plans. What information did the two statistics discard?

**Answer:** Replace this text with your explanation.

## 6 · When does 68–95–99.7 help? (10 min)
A normal model predicts approximately 68%, 95%, and 99.7% within 1, 2, and 3 SD of its mean. This is **not a guarantee for a finite dataset**, and symmetry alone is insufficient. We use sample mean/SD here to compare observed proportions with the model.

In [ ]:
rows=[]
for col in ['entry_score','weekly_minutes','assessment_score']:
    x=students[col]; m=x.mean(); sd=x.std(ddof=1)
    for k in [1,2,3]:
        rows.append({'column':col,'SD distance':k,'lower':m-k*sd,'upper':m+k*sd,
                     'observed %':100*x.between(m-k*sd,m+k*sd).mean(),
                     'normal model %':{1:68,2:95,3:99.7}[k]})
display(pd.DataFrame(rows))

### Your reasoning
**6A.** Give your entry-score interval within 1 SD and observed percentage. Explain why it need not be exactly 68%.

**6B.** Which column is the most plausible candidate for a normal approximation? Use both shape and proportions. Explain why applying the rule to either of the other columns could mislead.

**6C.** If an observed proportion happens to be near 95%, has normality been proven? Why not?

**Answer:** Replace this text with your explanation.

## 7 · Relative to whom? (15 min)
A z-score is `(value − reference mean) / reference SD`. It is unitless; the reference SD must be positive. Positive means above the mean, not automatically good. Computing z does not require normality; converting it into a normal-model rarity does.

**Prediction:** the selected student's raw score will stay fixed while the reference group changes. Must their z-score remain fixed? Explain before running.

### Your reasoning
**7A. Prediction:** explain which ingredients can change and why.

**Answer:** Replace this text with your explanation.

In [ ]:
focus=students.iloc[focus_index]
print('Selected record:',int(focus.record_id),'| programme:',focus.programme,'| entry score:',focus.entry_score)
rows=[]
for label,ref in [('Whole cohort',students.entry_score),
                  ('Programme A',students.loc[students.programme=='A','entry_score']),
                  ('Programme B',students.loc[students.programme=='B','entry_score'])]:
    sd=ref.std(ddof=1)
    rows.append({'reference':label,'n':len(ref),'mean':ref.mean(),'sample SD':sd,
                 'z':(focus.entry_score-ref.mean())/sd if sd>0 else np.nan})
display(pd.DataFrame(rows))
students['entry_z_cohort']=(students.entry_score-students.entry_score.mean())/students.entry_score.std(ddof=1)
print('Standardised column mean / sample SD:',students.entry_z_cohort.mean(),students.entry_z_cohort.std(ddof=1))
# A separate personalised example separates distance from rank.
rank_times = np.r_[np.arange(1,10),int(46 + seed % 21)].astype(float)
target=9.
print('Practice minutes:',rank_times.tolist())
print('Target:',target,'| z:',(target-rank_times.mean())/rank_times.std(ddof=1),
      '| percentile rank (at or below):',100*np.mean(rank_times<=target))

### Your reasoning
**7B.** Show one z-score calculation from the table. Write its interpretation with the reference group and direction. Which reference answers “unusual within this student's own programme”?

**7C.** Explain the mean and SD of the standardised column. Would standardising programme codes be meaningful? What if all values in a reference group were identical?

**7D.** In the practice-time example, how can 9 minutes be below the mean yet at the 90th percentile? Use the extreme observation in your explanation. Why should you not automatically interpret z=2 as a particular percentile?

**7E.** A bottle process has reference mean 100 mL and SD 2 mL. Its explicitly agreed specification is 94–106 mL inclusive. For bottles of 104 and 108 mL, calculate z and decide acceptance. Explain why rejecting a bottle does not mean deleting its measurement, and why z alone does not set the acceptance rule.

**Answer:** Replace this text with your explanation.

## 8 · Your recommendation (15 min)
The final case uses the same weekly-minute data. Your assigned request appears below. Choose your primary summary for **that decision**, then explain what you would report for the other decision.

- **Typical use:** describe the experience of a typical student.
- **Total workload:** estimate the recorded cohort's total platform workload for the week; distinguish an observed total from a forecast of a future week.

This is a decision exercise. There is no universally best average.

In [ ]:
print('YOUR REQUEST:',scenario)
x=students.weekly_minutes
print('Case:',case_code)
display(pd.Series({'valid n':x.count(),'mean':x.mean(),'median':x.median(),
    'sample SD':x.std(ddof=1),'Q1':x.quantile(.25),'Q3':x.quantile(.75),
    'IQR':x.quantile(.75)-x.quantile(.25),'total minutes':x.sum()}))

### Your reasoning
**8A. Final note — approximately 150 words:** explain why you would not report the weekly-minute mean alone. Address your assigned request and include: case code; population and valid n; units; chosen centre and spread; two numerical pieces of evidence; shape; what your summary hides; what you would report for the other request; and one limitation. Do not claim a cause or treat this synthetic cohort as real university evidence.

**8B. Transfer check:** suppose the highest-use student's minutes doubled. Without rerunning, predict the direction of change in the mean and total, and whether the median would change. Explain which recommendation would be more affected.

**Answer:** Replace this text with your explanation.

## Submission checklist
- [ ] My name, student ID, and case code are present and consistent.
- [ ] I kept predictions and explained any revisions.
- [ ] I answered 1A–8B, including the hand calculations and approximately 150-word note.
- [ ] Four column plots and all other supplied results are visible.
- [ ] My answers cite my own numbers, units, group definitions, and plot evidence.
- [ ] I stated the quartile convention and sample/population SD choice where relevant.
- [ ] I did not automatically remove outlier candidates or make unsupported causal claims.
- [ ] I restarted and ran all cells using my own ID, then saved with outputs.

Save as **`STUDENTID_Lab2_FDS.ipynb`**, replacing STUDENTID with your ID. Submit through the course's announced channel and follow its current email-subject rule. The reflection belongs inside this notebook; no separate report is needed.

### What a strong submission demonstrates
Correct calculations are the starting point. Strong work explains **why** a summary fits the question, distinguishes observations from assumptions, recognises what averages conceal, and supports a decision with personalised evidence.

*Aligned with Week 2: Describing Data, slides 2–65. This lab focuses on describing and interpreting data; full data-cleaning procedures follow in Week 3.*